# Equilibrium Aggregation on Graph Benchmarks

Node Classification on Cora: Robust graph representation learning using equilibrium-based median aggregation. This notebook implements the approach with `EquilibriumAggregation` inside a `K3EquilibriumNet` model, trained with the Adam optimizer for 10 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `EquilibriumAggregation` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import layers as k3_layers

title = "Equilibrium Aggregation on Graph Benchmark"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 2. Synthetic Data Generator for Median Learning
input_size = 100
batch_size = 8

# 1. Equilibrium Aggregation Model
class K3EquilibriumNet(keras.Model):
    def __init__(self, dim_size):
        super().__init__()
        self.aggr = k3_layers.EquilibriumAggregation(1, 1, [256, 256], grad_iter=5)
        # Keras's `fit`/`train_on_batch` runs one internal forward pass with
        # constant-filled placeholder tensors to validate the loss pipeline
        # before the real training step. If `dim_size` were inferred from
        # `index`'s values (e.g. `index.max() + 1`), that placeholder pass
        # would produce a different output size than the real batches and
        # the validation pass would raise a spurious shape-mismatch error.
        # Since every batch here always has exactly `batch_size` groups,
        # passing a fixed, static `dim_size` sidesteps that entirely.
        self.dim_size = dim_size

    def call(self, inputs, index=None):
        if isinstance(inputs, (tuple, list)):
            x, index = inputs[0], inputs[1]
        else:
            x = inputs
        return self.aggr(x, index=index, dim_size=self.dim_size)

k3_model = K3EquilibriumNet(dim_size=batch_size)

def data_generator():
    while True:
        x_list = []
        y_list = []
        index_list = []
        for b in range(batch_size):
            nums = np.random.uniform(-1, 1, size=(input_size, 1)).astype(np.float32)
            med = np.median(nums).astype(np.float32)
            x_list.append(nums)
            y_list.append(med)
            index_list.append(np.full((input_size,), b, dtype=np.int64))

        x_cat = np.concatenate(x_list, axis=0)
        index_cat = np.concatenate(index_list, axis=0)
        y_cat = np.array(y_list, dtype=np.float32).reshape(-1, 1)

        yield (x_cat, index_cat), y_cat

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.MeanSquaredError(),
    metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
)

# 4. Training
print(f"Training K3-Node EquilibriumAggregation on {backend} backend...")
history = k3_model.fit(
    data_generator(),
    steps_per_epoch=20,
    epochs=10,
    verbose=1,
)

print("\n✓ K3-Node execution completed successfully!")